In [ ]:
"""
Ultimate Faster-Whisper Transcription (LONG + MULTILINGUAL + AUTO CONVERT)

Fixes:
- Token repetition loop
- Multilingual drift
- Long audio collapse
- Bad audio format issues

Auto:
- Convert any media → WAV 16kHz mono
"""

import os
import time
import torch
import subprocess
from faster_whisper import WhisperModel

# =====================
# CONFIG
# =====================
INPUT_MEDIA = "AUHISTORYWW2.mp3"   # mp3 / m4a / mp4 / wav / etc
TMP_WAV = "tmp_whisper_input.wav"

MODEL_NAME = "large-v3-turbo"
OUT_PATH = "WW2_convert_trans001.txt"
SAVE_TXT = True

# Decoding (LONG + MULTILINGUAL SAFE)
BEAM_SIZE = 3
TEMPERATURE = [0.2, 0.4, 0.6]   # ❌ avoid 0.0
NO_SPEECH_THRESHOLD = 0.3
MIN_SILENCE_MS = 600

# =====================
# DEVICE
# =====================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE = "float16" if DEVICE == "cuda" else "int8"

print(f"Using device: {DEVICE} ({COMPUTE})")

# =====================
# AUTO CONVERT → WAV
# =====================
def convert_to_wav(input_path, output_path):
    print("🔄 Converting input media → WAV 16kHz mono...")
    cmd = [
        "ffmpeg", "-y",
        "-i", input_path,
        "-ac", "1",
        "-ar", "16000",
        "-vn",
        "-f", "wav",
        output_path
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    if not os.path.exists(output_path):
        raise RuntimeError("❌ Audio conversion failed")

# convert first
convert_to_wav(INPUT_MEDIA, TMP_WAV)

# =====================
# LOAD MODEL
# =====================
model = WhisperModel(
    MODEL_NAME,
    device=DEVICE,
    compute_type=COMPUTE,
    cpu_threads=os.cpu_count()
)

# =====================
# TRANSCRIBE
# =====================
print("\n🚀 Starting transcription (long multilingual safe)...\n")
start_time = time.time()

segments, info = model.transcribe(
    TMP_WAV,
    beam_size=BEAM_SIZE,
    temperature=TEMPERATURE,
    vad_filter=True,
    vad_parameters=dict(
        min_silence_duration_ms=MIN_SILENCE_MS
    ),
    no_speech_threshold=NO_SPEECH_THRESHOLD,
    condition_on_previous_text=False,   # 🔥 anti-loop
    multilingual=True
)

# =====================
# COLLECT TEXT (ANTI-LOOP FILTER)
# =====================
transcript_lines = []
last_text = ""

for seg in segments:
    text = seg.text.strip()

    if not text:
        continue

    if text == last_text:
        continue

    transcript_lines.append(text)
    last_text = text

transcript = " ".join(transcript_lines)

# =====================
# REPORT
# =====================
elapsed = time.time() - start_time

report = f"""
===== TRANSCRIPTION REPORT =====
Model       : {MODEL_NAME}
Input       : {INPUT_MEDIA}
Language    : {info.language} (auto-detect)
Duration    : {info.duration:.2f} sec
Segments    : {len(transcript_lines)}
Device      : {DEVICE}
Compute     : {COMPUTE}
Time used   : {elapsed:.2f} sec ({elapsed/60:.2f} min)
================================
"""

# =====================
# SAVE FILE
# =====================
if SAVE_TXT:
    with open(OUT_PATH, "w", encoding="utf-8") as f:
        f.write(report.strip() + "\n\n")
        f.write(transcript)

# =====================
# CLEANUP
# =====================
if os.path.exists(TMP_WAV):
    os.remove(TMP_WAV)

# =====================
# CONSOLE PREVIEW
# =====================
print(report)
print(transcript[:1200] + ("..." if len(transcript) > 1200 else ""))
print("\n✅ Transcription complete (AUTO CONVERT + NO LOOP)")
